# SDXL Dataset Generation — Kaggle Runner

Runs `generate_dataset.py` against COCO val2017 to produce the 1200-sample SDXL manipulation benchmark.

**One-time account setup (do this on kaggle.com before running):**
1. Verify your phone number under Settings — required to enable GPU + Internet on notebooks.
2. This notebook's settings (right sidebar): Accelerator -> GPU T4 x2 or P100, Internet -> On.
3. Upload this project as a private Kaggle Dataset (generate_dataset.py + configs/; skip data/coco,
   downloaded fresh in step 5) and attach it to this notebook via Add Input. Set INPUT_DATASET_SLUG
   below to match.

**Unlike Colab:** there is no Drive-FUSE mount here, so no chunked-checkpoint dance is needed --
everything lives on local notebook disk (/kaggle/working) for the session. For the real 1200-sample
run, use Save Version -> Save & Run All (Commit) instead of running cells interactively -- that runs
as a background batch job that keeps going even if you close the tab (session cap is ~9-12h GPU,
comfortably above our ~4.5h estimate).

## 1. Confirm a GPU is actually attached

In [ ]:
import subprocess

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True)
except FileNotFoundError:
    raise RuntimeError("nvidia-smi isn't installed - Accelerator is set to None. Open notebook settings (right sidebar), set Accelerator to GPU T4 x2 or P100, save, wait for restart, then re-run this cell.")

print(result.stdout.decode())
if result.returncode != 0:
    raise RuntimeError("nvidia-smi ran but reported an error. Check Accelerator is set to GPU T4 x2 or P100 in notebook settings, then re-run this cell.")
print("GPU check passed.")

## 2. Load project files into writable working storage

/kaggle/input/... (where attached datasets land) is read-only. Copy the code and config into
/kaggle/working so generate_dataset.py can run and write output there.

In [ ]:
import shutil
from pathlib import Path

INPUT_DATASET_SLUG = "dataset1"
INPUT_DIR = Path("/kaggle/input") / INPUT_DATASET_SLUG
PROJECT_DIR = "/kaggle/working/localization_project"

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"{INPUT_DIR} not found. Attach your uploaded dataset via Add Input and set INPUT_DATASET_SLUG above to its exact slug.")

def find_one(filename: str) -> Path:
    matches = list(INPUT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} anywhere under {INPUT_DIR}.")
    return matches[0]

script_src = find_one("generate_dataset.py")
config_src = find_one("generate_dataset_config.json")

project_path = Path(PROJECT_DIR)
(project_path / "configs").mkdir(parents=True, exist_ok=True)
shutil.copy2(script_src, project_path / "generate_dataset.py")
shutil.copy2(config_src, project_path / "configs" / "generate_dataset_config.json")

print("PROJECT_DIR OK:", PROJECT_DIR)
print("generate_dataset.py <-", script_src)
print("configs/generate_dataset_config.json <-", config_src)

## 3. Install dependencies

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors xformers tqdm

## 4. Verify the environment before touching the model

Fails fast with a clear message instead of a deep pipeline-load traceback.

In [ ]:
import torch
import diffusers

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("diffusers:", diffusers.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False - GPU accelerator not active. Fix step 1 first.")

try:
    import xformers
    print("xformers:", xformers.__version__)
except ImportError:
    print("xformers not importable - will disable enable_xformers in the config below.")

## 5. Download COCO val2017 (idempotent - skips if already present)

Uses unzip -n (never overwrite) so re-running this cell is always safe.

In [ ]:
import os

coco_dir = "/kaggle/working/coco"
images_dir = os.path.join(coco_dir, "val2017")
ann_file = os.path.join(coco_dir, "annotations", "instances_val2017.json")

os.makedirs(coco_dir, exist_ok=True)

if not os.path.isdir(images_dir) or len(os.listdir(images_dir)) < 5000:
    !wget -q -O /kaggle/working/val2017.zip http://images.cocodataset.org/zips/val2017.zip
    !unzip -n -q /kaggle/working/val2017.zip -d {coco_dir}
else:
    print("val2017 images already present, skipping download.")

if not os.path.isfile(ann_file):
    !wget -q -O /kaggle/working/annotations_trainval2017.zip http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -n -q /kaggle/working/annotations_trainval2017.zip -d {coco_dir}
else:
    print("annotations already present, skipping download.")

assert os.path.isdir(images_dir) and len(os.listdir(images_dir)) >= 5000, "val2017 images missing/incomplete"
assert os.path.isfile(ann_file), "instances_val2017.json missing"
print("COCO ready:", images_dir, "|", ann_file)

## 6. Update config for Kaggle

device must be "cuda" - torch_dtype/enable_xformers/enable_cpu_offload only take effect when
device == "cuda" in generate_dataset.py.

In [ ]:
import json

DATA_DIR = "/kaggle/working/data"
config_path = f"{PROJECT_DIR}/configs/generate_dataset_config.json"
with open(config_path) as f:
    cfg = json.load(f)

cfg["data_dir"] = DATA_DIR
cfg["coco_images_dir"] = images_dir
cfg["coco_annotations_file"] = ann_file
cfg["device"] = "cuda"
cfg["torch_dtype"] = "float16"
cfg["enable_xformers"] = True
cfg["enable_cpu_offload"] = True
cfg["dry_run"] = False

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

for key in ("data_dir", "coco_images_dir", "coco_annotations_file", "device", "torch_dtype",
            "enable_xformers", "enable_cpu_offload", "dry_run"):
    print(f"{key}: {cfg[key]}")

## 7. Smoke test - 3 real samples

Confirms the pipeline actually loads and generates on GPU before committing to the full 1200-image run.

In [ ]:
!cd {PROJECT_DIR} && python generate_dataset.py --config configs/generate_dataset_config.json --target-count 3 2>&1 | tee /kaggle/working/smoke_test.log

with open("/kaggle/working/smoke_test.log") as f:
    log = f.read()
if "Traceback" in log or "Fatal error" in log:
    raise RuntimeError("Smoke test failed - see /kaggle/working/smoke_test.log above for the traceback.")
print("Smoke test passed.")

## 8. Full run - 1200 samples

No chunking needed here (that was a Colab/Drive-specific workaround). generate_dataset.py's manifest
already resumes correctly if this cell is re-run.

**To run this unattended for ~4.5 hours without keeping the browser open:** don't just run this cell
interactively - use Save Version -> Save & Run All (Commit) from the top-right menu. That executes
the whole notebook as a background batch job on Kaggle's servers.

In [ ]:
!cd {PROJECT_DIR} && python generate_dataset.py --config configs/generate_dataset_config.json --target-count 1200 2>&1 | tee -a /kaggle/working/full_run.log

## 9. Verify and persist results

/kaggle/working only survives past this session if you Commit the notebook (its output becomes a new
Kaggle Dataset version automatically) - if you just close an interactive session without committing,
this is lost, same failure mode as Colab's /content.

In [ ]:
import json
from collections import Counter

with open(f"{DATA_DIR}/manifest.json") as f:
    manifest = json.load(f)

print("Manifest status counts:", dict(Counter(e["status"] for e in manifest.values())))
print("Total entries:", len(manifest))